# OceanStream QuickstartThis notebook demonstrates the fastest way to get started with **oceanstream** as a Python library.## What You'll Learn- Import the library- Get a data provider- Convert CSV data to GeoParquet in one function call

## Installation```bashpip install oceanstream```Or install with specific extras:```bashpip install oceanstream[geotrack]    # GPS/navigation processingpip install oceanstream[all]         # All features```

In [ ]:
# Quick setup for developmentimport sysfrom pathlib import Pathproject_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()if str(project_root) not in sys.path:    sys.path.insert(0, str(project_root))

## The Simplest WorkflowConvert CSV data to GeoParquet with just 3 lines of code:

In [ ]:
from oceanstream import get_provider, convert# Get a provider for your data sourceprovider = get_provider('saildrone')print(f"✅ Provider loaded: {provider.name}")print(f"   Supported modules: {provider.supported_modules}")

In [ ]:
import tempfile# Setup pathsinput_dir = project_root / "oceanstream" / "tests" / "data" / "raw_data"output_dir = Path(tempfile.mkdtemp()) / "output"print(f"📂 Input:  {input_dir}")print(f"📂 Output: {output_dir}")

In [ ]:
# Convert CSV to GeoParquet - one function call!convert(    provider=provider,    input_source=input_dir,    output_dir=output_dir,    campaign_id="quickstart_demo",    verbose=True,    yes=True,  # Skip confirmation prompts)

## Check the Results

In [ ]:
import pandas as pd# Find the generated parquet filescampaign_dir = output_dir / "quickstart_demo"parquet_files = list(campaign_dir.rglob("*.parquet"))print(f"📊 Generated {len(parquet_files)} partition files")# Read and display sampleif parquet_files:    df = pd.read_parquet(parquet_files[0])    print(f"📈 Sample data ({len(df)} rows):")    print(df[['time', 'latitude', 'longitude']].head())

## Output StructureThe `convert()` function creates:```output_dir/campaign_id/├── lat_bin=X/lon_bin=Y/*.parquet   # Hive-partitioned GeoParquet├── stac/collection.json             # STAC 1.0 metadata├── stac/items/*.json                # Per-partition STAC items└── .oceanstream_metadata.json       # Processing tracking```

In [ ]:
# Show output structuredef show_tree(path, prefix=""):    items = sorted(path.iterdir())    for i, item in enumerate(items[:10]):  # Limit output        is_last = i == len(items) - 1        print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}")        if item.is_dir() and len(list(item.iterdir())) < 20:            show_tree(item, prefix + ("    " if is_last else "│   "))print(f"📁 Output structure:")show_tree(campaign_dir)